# Linear Regression — Implementations

The same two estimators the notebook builds by hand, in the lanes a working practitioner would reach for. Step names match `first_principles.ipynb` so the side-by-side view lines up.

## 01_ols_closed_form

Least squares in one shot.

### torch

`torch.linalg.lstsq` solves the least-squares problem by QR rather than by forming `XᵀX`. **What torch adds:** numerical stability — the normal equations square the condition number, and QR does not.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Add the intercept as a column of ones, exactly as the NumPy lane does.
# 2. torch.linalg.lstsq solves least squares by QR, not via the normal equations.
# 3. Work in float64 — float32 loses ~7 digits and the lanes stop agreeing.
# 4. .solution is (p+1, 1) for one right-hand side; squeeze it back to a vector.


class LinearRegressionOLS:
    """Ordinary least squares, solved by torch's own least-squares routine."""

    def fit(self, X, y):
        Xt = torch.as_tensor(np.asarray(X, dtype=np.float64))
        yt = torch.as_tensor(np.asarray(y, dtype=np.float64))
        Xd = torch.cat([torch.ones(Xt.shape[0], 1, dtype=Xt.dtype), Xt], dim=1)
        theta = torch.linalg.lstsq(Xd, yt.unsqueeze(1)).solution.squeeze(1)
        self.theta_ = theta
        self.intercept_ = float(theta[0])
        self.coef_ = theta[1:].numpy()
        return self

    def predict(self, X):
        Xt = torch.as_tensor(np.asarray(X, dtype=np.float64))
        return (self.intercept_ + Xt @ self.theta_[1:]).numpy()


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(11)
X_eq = _rng_eq.normal(size=(80, 3))
y_eq = X_eq @ np.array([2.0, -1.0, 0.5]) + 4.0 + _rng_eq.normal(0, 0.1, size=80)

_fit_eq = LinearRegressionOLS().fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("intercept:", round(intercept, 6))
print("coef     :", np.round(coef, 6))


In [ ]:
assert coef.shape == (3,), "one coefficient per feature, intercept held apart"
assert abs(intercept - 4.0) < 0.05, "the intercept should recover the 4.0 it was built from"

_truth = np.array([2.0, -1.0, 0.5])
assert np.max(np.abs(coef - _truth)) < 0.05, "coefficients should recover the generating vector"

# The point of a closed form: the residual is orthogonal to every column.
_resid = y_eq - _fit_eq.predict(X_eq)
assert abs(_resid.sum()) < 1e-8, "residuals sum to zero when an intercept is fitted"
assert np.max(np.abs(X_eq.T @ _resid)) < 1e-8, "residuals are orthogonal to the design"


### library

What you would actually write. **What the library adds:** nothing you did not just derive — which is the point of having derived it.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# hints:
# 1. fit_intercept=True centres X and y for you; do not add a ones column too.
# 2. coef_ excludes the intercept, which lives in intercept_ — same split as ours.
# 3. sklearn solves the same least-squares problem, so expect agreement to ~1e-12.


class LinearRegressionOLS:
    """The two-line version. `fit_intercept=True` centres for you."""

    def fit(self, X, y):
        self._model = LinearRegression().fit(np.asarray(X), np.asarray(y))
        self.coef_ = self._model.coef_
        self.intercept_ = float(self._model.intercept_)
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X))


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(11)
X_eq = _rng_eq.normal(size=(80, 3))
y_eq = X_eq @ np.array([2.0, -1.0, 0.5]) + 4.0 + _rng_eq.normal(0, 0.1, size=80)

_fit_eq = LinearRegressionOLS().fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("intercept:", round(intercept, 6))


In [ ]:
assert coef.shape == (3,)
_resid = y_eq - LinearRegressionOLS().fit(X_eq, y_eq).predict(X_eq)
assert np.max(np.abs(X_eq.T @ _resid)) < 1e-8, "sklearn solves the same normal equations"


## 01_linear_regression_gd

The same fit, walked downhill.

### torch

Identical arithmetic to the NumPy lane — zeros init, `theta -= lr * grad`, the same number of steps — with one line changed. **What torch adds:** `loss.backward()` in place of the hand-derived `(2/n) Xᵀr`; the check below shows they are the same numbers.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Standardise X first; the workable learning rate belongs to that problem.
# 2. Start theta at zeros, with requires_grad=True so autograd tracks it.
# 3. backward() fills theta.grad with exactly (2/n) Xd.T @ resid — check it once.
# 4. Clear theta.grad every step, or torch accumulates it across iterations.
# 5. Map the standardised weights back to the original scale at the end.


class LinearRegressionGD:
    """The same descent, with autograd supplying the gradient."""

    def __init__(self, lr=0.1, n_iter=500, tol=1e-8):
        self.lr = lr
        self.n_iter = n_iter
        self.tol = tol

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        n, p = X.shape
        self.mu_ = X.mean(axis=0)
        self.sigma_ = X.std(axis=0)
        self.sigma_[self.sigma_ == 0] = 1.0
        Xs = (X - self.mu_) / self.sigma_
        Xd = torch.as_tensor(np.column_stack([np.ones(n), Xs]))
        yt = torch.as_tensor(np.asarray(y, dtype=float))

        theta = torch.zeros(p + 1, dtype=torch.float64, requires_grad=True)
        self.loss_history_ = []
        for _ in range(self.n_iter):
            loss = torch.mean((Xd @ theta - yt) ** 2)
            loss.backward()
            with torch.no_grad():
                theta -= self.lr * theta.grad
                grad_norm = float(torch.linalg.norm(theta.grad))
                theta.grad = None
            self.loss_history_.append(float(torch.mean((Xd @ theta - yt) ** 2)))
            if grad_norm < self.tol:
                break

        theta = theta.detach().numpy()
        b_std, w_std = theta[0], theta[1:]
        self.coef_ = w_std / self.sigma_
        self.intercept_ = float(b_std - np.sum(w_std * self.mu_ / self.sigma_))
        self.theta_ = np.r_[self.intercept_, self.coef_]
        self.loss_history_ = np.array(self.loss_history_)
        return self

    def predict(self, X):
        return self.intercept_ + np.asarray(X) @ self.coef_


In [ ]:
# exports: coef, intercept
_rng_eq = np.random.default_rng(11)
X_eq = _rng_eq.normal(size=(80, 3))
y_eq = X_eq @ np.array([2.0, -1.0, 0.5]) + 4.0 + _rng_eq.normal(0, 0.1, size=80)

_fit_eq = LinearRegressionGD(lr=0.3, n_iter=3000, tol=0.0).fit(X_eq, y_eq)
coef, intercept = _fit_eq.coef_, _fit_eq.intercept_
print("final loss:", round(float(_fit_eq.loss_history_[-1]), 8))


In [ ]:
assert _fit_eq.loss_history_[-1] < _fit_eq.loss_history_[0], "the loss has to go down"
assert np.all(np.diff(_fit_eq.loss_history_) <= 1e-12), "and monotonically, at this step size"

# Autograd's gradient is the hand-derived one, not an approximation of it.
_n = X_eq.shape[0]
_Xs = (X_eq - X_eq.mean(axis=0)) / X_eq.std(axis=0)
_Xd = torch.as_tensor(np.column_stack([np.ones(_n), _Xs]))
_yt = torch.as_tensor(y_eq)
_th = torch.zeros(4, dtype=torch.float64, requires_grad=True)
torch.mean((_Xd @ _th - _yt) ** 2).backward()
_by_hand = (2.0 / _n) * _Xd.T @ (_Xd @ _th.detach() - _yt)
assert torch.allclose(_th.grad, _by_hand, atol=1e-12), "autograd == the derivation"
